In [5]:
import sqlite3
import pandas as pd
from loguru import logger
import os

# Aparte tabellen die we gaan gebruiken
conn1 = sqlite3.connect('BikeToDrive_1_Accessoireverkoop.db')
conn2 = sqlite3.connect('BikeToDrive_2_Fietsverkoop.db')
conn3 = sqlite3.connect('BikeToDrive_3_Onderhoud.db')
conn4 = sqlite3.connect('BikeToDrive_4_Accessoire_Inkoop.db')
conn5 = sqlite3.connect('BikeToDrive_5_Fiets_Inkoop.db')

# Een file waar alles erin staat
conn = sqlite3.connect('SDM.db')

In [6]:
def reset_sdm():
    logger.info("Start full refresh...")
    try:
        cursor = conn.cursor()

        """ Haal alle tabellen op
        We moeten dan per tabel oproepen"""

        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tabellen = []

        # De data wordt binnen de 'tabellen' lijst gestopt.
        for tabel in cursor.fetchall():
            tabellen.append(tabel[0])

        # Op basis van deze tabellen lijst wordt de data binnen het tabel verwijderd.
        for tabel in tabellen:
            cursor.execute(f"DELETE FROM {tabel}")

        conn.commit()
        conn.close()
        print("Reset geslaagd! Alle tabellen zijn leeg.")
        
    except Exception as e:
        print(f"Fout bij reset: {e}")

reset_sdm()

2026-03-23 15:44:07.376 | INFO     | __main__:reset_sdm:2 - Start full refresh...


Reset geslaagd! Alle tabellen zijn leeg.


In [ ]:
# Wij maken gebruik van prefix, omdat het anders bij ons niet werkt

def bepaal_prefix(bron_db):
    # haal bestandsnaam eruit
    naam = bron_db.replace("BikeToDrive_", "").replace(".db", "")

    # verwijder nummer bijvoorbeeld 1_, 2_, enz.
    naam = naam.split("_", 1)[1]

    # kleine correcties voor naming
    naam = naam.replace("Accessoireverkoop", "Accessoire_Verkoop")
    naam = naam.replace("Fietsverkoop", "Fietsverkoop")
    naam = naam.replace("Accessoire_inkoop", "Accessoire_Inkoop")
    naam = naam.replace("Fiets_Inkoop", "Fiets_Inkoop")

    return naam

In [8]:
# Nadat de Full refresh uitgevoerd is, maken we gebruik van ''
def overzetten_alle_data():
    sdm_pad = 'SDM.db'
    logger.info("Start schema-driven bulk loading...")

    # De volledige lijst met alle bronnen en doeltabellen
    prefix_map = {
        'BikeToDrive_1_Accessoireverkoop.db': 'Accessoire_Verkoop',
        'BikeToDrive_2_Fietsverkoop.db': 'Fietsverkoop',
        'BikeToDrive_3_Onderhoud.db': 'Onderhoud',
        'BikeToDrive_4_Accessoire_inkoop.db': 'Accessoire_Inkoop',
        'BikeToDrive_5_Fiets_Inkoop.db': 'Fiets_Inkoop'
    }

    conn_sdm = sqlite3.connect(sdm_pad)

      # Loop over databases
    for bron_db in prefix_map:

        if not os.path.exists(bron_db):
            logger.error(f"Bestand niet gevonden: {bron_db}")
            continue

        prefix = prefix_map[bron_db]

        conn_bron = sqlite3.connect(bron_db)
        cursor = conn_bron.cursor()

        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

        tabellen = []
        for t in cursor.fetchall():
            tabellen.append(t[0])

        # Loop over tabellen
        for tabel in tabellen:
            if tabel == "sqlite_sequence":
                continue
            
            try:
                df = pd.read_sql_query(f"SELECT * FROM {tabel}", conn_bron)

                # We maken gebruik vand de prefix zodat de tabelnamen goed staan
                if "Verkoop" in tabel or "Inkoop" in tabel or tabel == "Onderhoud":
                    doel_tabel = tabel
                else:
                    doel_tabel = f"{prefix}_{tabel}"

                df.to_sql(doel_tabel, conn_sdm, if_exists='replace', index=False)

                logger.success(f"{doel_tabel}: {len(df)} rijen geladen")

            except Exception as e:
                logger.warning(f"Fout bij {tabel}: {e}")
        # Nadat de Schema Bulk Loading uitgevoerd is, wordt de connectie bron_db afgesloten
        conn_bron.close()

    conn_sdm.close()
    logger.success("Schema-driven loading compleet!")

# =============== Hieronder wordt de methode uitgevoerd ===============

overzetten_alle_data()

2026-03-23 15:44:14.960 | INFO     | __main__:overzetten_alle_data:4 - Start schema-driven bulk loading...
2026-03-23 15:44:14.989 | SUCCESS  | __main__:overzetten_alle_data:51 - Accessoire_Verkoop: 100 rijen geladen
2026-03-23 15:44:15.017 | SUCCESS  | __main__:overzetten_alle_data:51 - Accessoire_Verkoop_Monteur: 10 rijen geladen
2026-03-23 15:44:15.043 | SUCCESS  | __main__:overzetten_alle_data:51 - Accessoire_Verkoop_Leverancier: 5 rijen geladen
2026-03-23 15:44:15.073 | SUCCESS  | __main__:overzetten_alle_data:51 - Accessoire_Verkoop_Accessoire: 10 rijen geladen
2026-03-23 15:44:15.099 | SUCCESS  | __main__:overzetten_alle_data:51 - Accessoire_Verkoop_Filiaal: 4 rijen geladen
2026-03-23 15:44:15.125 | SUCCESS  | __main__:overzetten_alle_data:51 - Accessoire_Verkoop_Klant: 20 rijen geladen
2026-03-23 15:44:15.151 | SUCCESS  | __main__:overzetten_alle_data:51 - Fiets_Verkoop: 150 rijen geladen
2026-03-23 15:44:15.175 | SUCCESS  | __main__:overzetten_alle_data:51 - Fietsverkoop_Fiets

In [ ]:
# """ Mapping Driven Loading - (Backup)"""

# # Nadat de Full refresh uitgevoerd is, maken we gebruik van ''
# def overzetten_alle_data():
#     sdm_pad = 'SDM.db'
#     logger.info("Start van de volledige data-pomp naar SDM...")

#     # De volledige lijst met alle bronnen en doeltabellen
#     mapping_config = {
#         'BikeToDrive_1_Accessoireverkoop.db': {
#             'Klant': 'Accessoire_Verkoop_Klant', 
#             'Filiaal': 'Accessoire_Verkoop_Filiaal',
#             'Monteur': 'Accessoire_Verkoop_Monteur', 
#             'Leverancier': 'Accessoire_Verkoop_Leverancier',
#             'Accessoire': 'Accessoire_Verkoop_Accessoire', 
#             'Accessoire_Verkoop': 'Accessoire_Verkoop' 
#         },
#         'BikeToDrive_2_Fietsverkoop.db': {
#             'Klant': 'Fietsverkoop_Klant', 
#             'Filiaal': 'Fietsverkoop_Filiaal', 
#             'Fiets': 'Fietsverkoop_Fiets',  
#             'Fiets_Verkoop': 'Fiets_Verkoop' 
#         },
#         'BikeToDrive_3_Onderhoud.db': {
#             'Fabrikant': 'Onderhoud_Fabrikant', 
#             'Fiets': 'Onderhoud_Fiets',
#             'Filiaal': 'Onderhoud_Filiaal', 
#             'Monteur': 'Onderhoud_Monteur', 
#             'Onderhoud': 'Onderhoud'
#         },
#         'BikeToDrive_4_Accessoire_inkoop.db': {
#             'Leverancier': 'Accessoire_Inkoop_Leverancier', 
#             'Accessoire': 'Accessoire_Inkoop_Accessoire',
#             'Accessoire_Inkoop': 'Accessoire_Inkoop'
#         },
#         'BikeToDrive_5_Fiets_Inkoop.db': {
#             'Fabrikant': 'Fiets_Inkoop_Fabrikant', 
#             'Fiets': 'Fiets_Inkoop_Fiets',
#             'Fiets_Inkoop': 'Fiets_Inkoop'
#         }
#     }

# # De data wordt dan terug erin toegevoegd
#     try:
#         conn_sdm = sqlite3.connect(sdm_pad)
        
#         for bron_db, tabellen in mapping_config.items():
#             if not os.path.exists(bron_db):
#                 logger.error(f"Bestand niet gevonden: {bron_db}")
#                 continue
                
#             conn_bron = sqlite3.connect(bron_db)
#             logger.debug(f"Bezig met bronbestand: {bron_db}")
            
#             for bron_tabel, sdm_tabel in tabellen.items():
#                 try:
#                     df = pd.read_sql_query(f"SELECT * FROM {bron_tabel}", conn_bron)
#                     # We gebruiken 'append' zodat de tabellen gevuld worden
#                     df.to_sql(sdm_tabel, conn_sdm, if_exists='append', index=False)
#                     logger.success(f"Gekopieerd: {len(df)} rijen naar {sdm_tabel}")
#                 except Exception as e:
#                     logger.warning(f"Kon {bron_tabel} niet laden: {e}")
            
#             conn_bron.close()
            
#         conn_sdm.close()
#         logger.success("Klaar! Alle beschikbare data is overgezet naar het SDM.")

#     except Exception as e:
#         logger.critical(f"Het proces is gestopt door een fout: {e}")

# # DIT IS DE BELANGRIJKSTE REGEL: Hiermee start je de codessss

# overzetten_alle_data()

2026-03-23 14:59:40.800 | INFO     | __main__:overzetten_alle_data:6 - Start van de volledige data-pomp naar SDM...
2026-03-23 14:59:40.806 | DEBUG    | __main__:overzetten_alle_data:53 - Bezig met bronbestand: BikeToDrive_1_Accessoireverkoop.db
2026-03-23 14:59:40.820 | SUCCESS  | __main__:overzetten_alle_data:60 - Gekopieerd: 20 rijen naar Accessoire_Verkoop_Klant
2026-03-23 14:59:40.834 | SUCCESS  | __main__:overzetten_alle_data:60 - Gekopieerd: 4 rijen naar Accessoire_Verkoop_Filiaal
2026-03-23 14:59:40.848 | SUCCESS  | __main__:overzetten_alle_data:60 - Gekopieerd: 10 rijen naar Accessoire_Verkoop_Monteur
2026-03-23 14:59:40.861 | SUCCESS  | __main__:overzetten_alle_data:60 - Gekopieerd: 5 rijen naar Accessoire_Verkoop_Leverancier
2026-03-23 14:59:40.873 | SUCCESS  | __main__:overzetten_alle_data:60 - Gekopieerd: 10 rijen naar Accessoire_Verkoop_Accessoire
2026-03-23 14:59:40.887 | SUCCESS  | __main__:overzetten_alle_data:60 - Gekopieerd: 100 rijen naar Accessoire_Verkoop
2026-03-